In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/home/dell/ML-Learning/datasets/online_retail_II_cleaned.csv")
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
print(df.shape)
df.head()

(1021452, 10)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancelled,has_customer_id
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,True


In [3]:
def add_time_features(df):
    df["year"] = df["invoicedate"].dt.year
    df["month"] = df["invoicedate"].dt.month
    df["month_name"] = df["invoicedate"].dt.month_name()
    df["quarter"] = df["invoicedate"].dt.quarter
    df["day"] = df["invoicedate"].dt.day
    df["day_of_week"] = df["invoicedate"].dt.day_name()
    df["hour"] = df["invoicedate"].dt.hour
    df["is_weekend"] = df["invoicedate"].dt.dayofweek >= 5
    df["year_month"] = df["invoicedate"].dt.to_period("M").astype(str)

    # rough part-of-day bucket, useful for checking when people shop
    def part_of_day(hour):
        if hour < 12:
            return "morning"
        elif hour < 17:
            return "afternoon"
        else:
            return "evening"
    df["day_part"] = df["hour"].apply(part_of_day)

    # Nov/Dec = holiday shopping season for this dataset (UK retailer)
    df["is_holiday_season"] = df["month"].isin([11, 12])
    return df

df = add_time_features(df)

In [4]:
def add_transaction_features(df):
    df["total_price"] = df["quantity"] * df["price"]

    # basket-level aggregates (same value repeated across every line in an invoice)
    df["items_in_invoice"] = df.groupby("invoice")["stockcode"].transform("nunique")
    df["total_units_in_invoice"] = df.groupby("invoice")["quantity"].transform("sum")
    df["invoice_total_value"] = df.groupby("invoice")["total_price"].transform("sum")

    # simple order-size buckets, easier to group by than raw revenue
    df["order_value_tier"] = pd.cut(
        df["invoice_total_value"],
        bins=[-np.inf, 20, 100, 500, np.inf],
        labels=["small", "medium", "large", "bulk"]
    )

    # a bulk order flag - useful given we found genuine wholesale buyers earlier
    df["is_bulk_order"] = df["quantity"] >= 100

    df["is_domestic"] = df["country"] == "United Kingdom"
    return df

df = add_transaction_features(df)

In [5]:
def build_customer_features(df):
    valid = df[df["has_customer_id"] & ~df["is_cancelled"]]
    snapshot_date = df["invoicedate"].max() + pd.Timedelta(days=1)

    customer_features = valid.groupby("customer_id").agg(
        recency_days=("invoicedate", lambda x: (snapshot_date - x.max()).days),
        frequency=("invoice", "nunique"),
        monetary=("total_price", "sum"),
        avg_order_value=("invoice_total_value", "mean"),
        first_purchase=("invoicedate", "min"),
        last_purchase=("invoicedate", "max"),
        unique_products_bought=("stockcode", "nunique"),
        countries_ordered_from=("country", "nunique"),
    )
    num_cols = customer_features.select_dtypes(include="number").columns
    customer_features[num_cols] = customer_features[num_cols].round(2)

    customer_features["customer_tenure_days"] = (
        customer_features["last_purchase"] - customer_features["first_purchase"]
    ).dt.days

    customer_features["avg_days_between_orders"] = (
        customer_features["customer_tenure_days"] / customer_features["frequency"]
    ).replace([np.inf, -np.inf], np.nan).round(1)

    customer_features["is_repeat_customer"] = customer_features["frequency"] > 1

    return customer_features.sort_values("monetary", ascending=False)

customer_features = build_customer_features(df)
print(customer_features.shape)
customer_features.head(10)

(5852, 11)


,recency_days,frequency,monetary,avg_order_value,first_purchase,last_purchase,unique_products_bought,countries_ordered_from,customer_tenure_days,avg_days_between_orders,is_repeat_customer
customer_id,,,,,,,,,,,
18102.0,1,145,580987.04,5970.20,2009-12-01 09:24:00,2011-12-09 11:50:00,382,1,738,5.1,True
14646.0,2,145,526751.52,7849.38,2009-12-02 16:52:00,2011-12-08 12:12:00,960,1,735,5.1,True
14156.0,10,144,303069.88,3458.02,2009-12-01 12:30:00,2011-11-30 10:54:00,1443,1,728,5.1,True
14911.0,1,373,272252.79,1051.92,2009-12-01 11:41:00,2011-12-08 15:54:00,2546,1,737,2.0,True
17450.0,8,51,244784.25,6431.79,2010-09-27 16:59:00,2011-12-01 13:29:00,144,1,429,8.4,True
13694.0,4,143,195640.69,3014.81,2009-12-04 15:26:00,2011-12-06 09:32:00,896,1,731,5.1,True
17511.0,3,60,172132.87,3882.25,2009-12-02 10:52:00,2011-12-07 10:12:00,657,1,734,12.2,True
16446.0,1,2,168472.50,56158.47,2011-05-18 09:52:00,2011-12-09 09:15:00,3,1,204,102.0,True
16684.0,4,55,147142.77,5210.30,2009-12-07 12:56:00,2011-12-05 14:06:00,184,1,728,13.2,True


In [6]:
def build_product_features(df):
    valid = df[~df["is_cancelled"]]

    product_features = valid.groupby(["stockcode", "description"]).agg(
        total_quantity_sold=("quantity", "sum"),
        total_revenue=("total_price", "sum"),
        num_orders=("invoice", "nunique"),
        num_countries_sold_to=("country", "nunique"),
        avg_price=("price", "mean"),
    ).round(2)

    product_features["revenue_per_order"] = (
        product_features["total_revenue"] / product_features["num_orders"]
    ).round(2)

    return product_features.sort_values("total_revenue", ascending=False)

product_features = build_product_features(df)
print(product_features.shape)
product_features.head(10)

(5587, 6)


,,total_quantity_sold,total_revenue,num_orders,num_countries_sold_to,avg_price,revenue_per_order
stockcode,description,,,,,,
22423,REGENCY CAKESTAND 3 TIER,26478,330590.32,3918,31,14.19,84.38
85123A,WHITE HANGING HEART T-LIGHT HOLDER,94142,257546.20,5356,23,3.08,48.09
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1,1,2.08,168469.60
47566,PARTY BUNTING,28200,148318.28,2674,21,5.72,55.47
85099B,JUMBO BAG RED RETROSPOT,77280,145961.83,3245,21,2.37,44.98
84879,ASSORTED COLOUR BIRD ORNAMENT,80082,129324.49,2807,19,1.86,46.07
22086,PAPER CHAIN KIT 50'S CHRISTMAS,35084,117760.29,2018,12,3.37,58.35
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247,10,1.47,330.77
79321,CHILLI LIGHTS,15841,80540.88,1135,10,6.29,70.96


In [7]:
print(df.columns.tolist())
df[["total_price", "day_part", "order_value_tier", "is_bulk_order", "is_domestic"]].head()

['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'is_cancelled', 'has_customer_id', 'year', 'month', 'month_name', 'quarter', 'day', 'day_of_week', 'hour', 'is_weekend', 'year_month', 'day_part', 'is_holiday_season', 'total_price', 'items_in_invoice', 'total_units_in_invoice', 'invoice_total_value', 'order_value_tier', 'is_bulk_order', 'is_domestic']


,total_price,day_part,order_value_tier,is_bulk_order,is_domestic
0,83.4,morning,bulk,False,True
1,81.0,morning,bulk,False,True
2,81.0,morning,bulk,False,True
3,100.8,morning,bulk,False,True
4,30.0,morning,bulk,False,True


In [8]:
df.to_csv("/home/dell/ML-Learning/datasets/online_retail_II_features.csv", index=False)
customer_features.to_csv("/home/dell/ML-Learning/datasets/customer_features.csv")
product_features.to_csv("/home/dell/ML-Learning/datasets/product_features.csv")
print("Saved feature-engineered datasets.")

Saved feature-engineered datasets.
